# Overview: Prediction Browsers & Cross-Method Comparisons

This notebook lets you:

- **Browse** all denoised TIFFs produced by your checkpoints for each element.
- **Compare** any two *alternative methods* against a **specific checkpoint** output side-by-side.
- Keep contrast consistent via *joint contrast* to make visual differences easier to spot.

---

## Copyright & License

**© 2025 Dmitry Karpov**  
Assistant Professor of Physics and Materials Science, Université Grenoble Alpes  
(Materials Modelling and Exploration Laboratory, MEM – UGA/CEA)

These notebooks were created as **educational material** for the  
**DIADEM Academy training series on U-Net and image analysis** (PEPR DIADEM)

Unless otherwise stated, all notebook content is distributed under the:

**Creative Commons Attribution–NonCommercial 4.0 International License (CC BY-NC 4.0)**  
https://creativecommons.org/licenses/by-nc/4.0/

You are free to **use, share, adapt, and modify** this material for **non-commercial research and teaching**,  
provided that **proper attribution** is given to the author.

Commercial use is **not permitted** without prior written permission.

### Please cite when using or adapting this material:
D. Karpov, *U-Net for Image Analysis — DIADEM Academy Training Materials*, Université Grenoble Alpes, 2025-present.

### Disclaimer
This material is provided *“as is”*, without warranty of any kind.  
The author and associated institutions are not liable for any damages arising from its use.

---

In [ ]:
# this cell is needed to mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, re, numpy as np
import matplotlib.pyplot as plt
from ipywidgets import (
    IntSlider, Dropdown, Text, Button, Checkbox, HBox, VBox, HTML, Output, Layout
)
from IPython.display import display
from mpl_toolkits.axes_grid1 import make_axes_locatable

try:
    import tifffile
except Exception as e:
    raise RuntimeError("tifffile is required to load .tif images.") from e

# If you're on Colab, enable the custom widget manager (safe no-op elsewhere)
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

# The path to the base folder
base_path = Path("/content/drive/MyDrive/setkafluo_demo/")

TRAIN_BASE = base_path / "training/" # path to pretrained models
DATA_DIR  = base_path / "input_data/" # data path
COMP_BASE  = DATA_DIR / "comparisons/"

# Element-specific prediction directories
PRED_DIRS = {
    "K-K":  TRAIN_BASE / "K-K"  / "predictions",
    "Ir-L": TRAIN_BASE / "Ir-L" / "predictions",
    "Zn-K": TRAIN_BASE / "Zn-K" / "predictions",
}

# Element-specific comparison directories (alternative methods)
COMP_DIRS = {
    "K":  COMP_BASE / "K",
    "Ir": COMP_BASE / "Ir",
    "Zn": COMP_BASE / "Zn",
}

# ---------- Helpers ----------
def list_tiffs(folder: Path):
    """Return a sorted list of .tif/.tiff Paths in folder (sorted by name)."""
    if not folder.exists():
        return []
    files = sorted([p for p in folder.iterdir() if p.suffix.lower() in {".tif", ".tiff"}])
    return files

_epoch_re = re.compile(r"(\d+)(?=\.tif{1,2}$)", re.IGNORECASE)

def sort_predictions_by_epoch(files):
    """
    If names look like '..._epoch_00025.tif' or end with digits, sort by that number.
    Otherwise return name-sorted list.
    """
    def key(p):
        m = _epoch_re.search(p.name)
        return (0, int(m.group(1))) if m else (1, p.name.lower())
    return sorted(files, key=key)

def load_tiff(path: Path) -> np.ndarray:
    """Load TIFF as float32 NumPy array."""
    arr = np.array(tifffile.imread(str(path)))
    return arr.astype(np.float32, copy=False)

def percentiles(img: np.ndarray, p=(1, 99)):
    """Safe 1–99% stretch with fallback to min/max for degenerate images."""
    p1, p99 = np.percentile(img, list(p))
    if p1 == p99:
        p1, p99 = float(np.min(img)), float(np.max(img))
    return p1, p99

def show_image(ax, img, vmin, vmax, title=""):
    """imshow + slim colorbar + correct aspect."""
    im = ax.imshow(img, vmin=vmin, vmax=vmax, cmap=None)  # default colormap
    ax.set_title(title, fontsize=11)
    ax.set_axis_off()
    ax.set_box_aspect(img.shape[0] / img.shape[1])
    cax = make_axes_locatable(ax).append_axes("right", size="3%", pad=0.05)
    ax.figure.colorbar(im, cax=cax)

# ---------- Browsers ----------
def create_predictions_browser(pred_dir: Path, element_label: str):
    """
    Interactive browser for a directory of predicted TIFFs.
    Displays a single image and lets you scroll through files.
    """
    files = sort_predictions_by_epoch(list_tiffs(pred_dir))
    info  = HTML()
    out   = Output()

    if not files:
        display(HTML(f"<b>No TIFFs found in:</b> <code>{pred_dir}</code>"))
        return

    slider = IntSlider(value=0, min=0, max=len(files)-1, step=1,
                       description="file", continuous_update=False, layout=Layout(width="650px"))
    btn_prev = Button(description="⟵ Prev", layout=Layout(width="90px"))
    btn_next = Button(description="Next ⟶", layout=Layout(width="90px"))
    joint_box = HBox([btn_prev, slider, btn_next], layout=Layout(gap="8px"))

    def draw(i: int):
        out.clear_output(wait=True)
        path = files[i]
        img  = load_tiff(path)
        vmin, vmax = percentiles(img)
        with out:
            fig, ax = plt.subplots(figsize=(7.5, 7.5))
            show_image(ax, img, vmin, vmax, title=f"{element_label} — {path.name}")
            fig.tight_layout()
            plt.show()
        info.value = f"<code>{i+1}/{len(files)}</code> &nbsp; {path.name} &nbsp; <span style='color:#555'>{pred_dir}</span>"

    def on_prev(_):
        if slider.value > slider.min:
            slider.value -= 1

    def on_next(_):
        if slider.value < slider.max:
            slider.value += 1

    def on_slide(change):
        if change.get("name") == "value":
            draw(int(change["new"]))

    slider.observe(on_slide, names="value")
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)

    display(VBox([joint_box, info, out]))
    draw(slider.value)

# ---------- Comparisons ----------
def create_comparison_view(element_short: str, comp_dir: Path, pred_dir: Path, title_prefix: str):
    """
    Compare 3 images side-by-side:
      - A: selectable from 'comp_dir'
      - B: selectable from 'comp_dir'
      - C: always the 'checkpoint' from 'pred_dir' (user types checkpoint filename)
    """
    comp_files = list_tiffs(comp_dir)
    pred_files = list_tiffs(pred_dir)

    if not comp_files:
        display(HTML(f"<b>No comparison TIFFs found in:</b> <code>{comp_dir}</code>"))
    if not pred_files:
        display(HTML(f"<b>No checkpoint TIFFs found in:</b> <code>{pred_dir}</code>"))

    # Dropdowns for A, B
    dd_A = Dropdown(
        options=[(p.name, str(p)) for p in comp_files],
        value=str(comp_files[0]) if comp_files else None,
        description="A:", layout=Layout(width="480px")
    )
    dd_B = Dropdown(
        options=[(p.name, str(p)) for p in comp_files],
        value=str(comp_files[min(1, max(0, len(comp_files)-1))]) if comp_files else None,
        description="B:", layout=Layout(width="480px")
    )

    # Text input for checkpoint filename (user-typed)
    suggested = pred_files[-1].name if pred_files else ""
    cp_text = Text(
        value=suggested,
        placeholder="prediction_epoch_00025.tif",
        description="Checkpoint:",
        layout=Layout(width="480px")
    )

    # (Optional) drop-down of suggestions to help typing
    dd_suggest = Dropdown(
        options=[(p.name, p.name) for p in sort_predictions_by_epoch(pred_files)],
        value=suggested if suggested else None,
        description="Suggest:",
        layout=Layout(width="480px")
    )

    def sync_from_suggest(change):
        if change.get("name") == "value" and change["new"]:
            cp_text.value = change["new"]
    dd_suggest.observe(sync_from_suggest, names="value")

    joint_contrast = Checkbox(value=True, description="Joint contrast", indent=False)
    refresh_btn    = Button(description="Refresh", layout=Layout(width="110px"))
    info = HTML()
    out  = Output()

    def draw(*_):
        out.clear_output(wait=True)
        # Load A/B (if available)
        imA = load_tiff(Path(dd_A.value)) if dd_A.value and os.path.exists(dd_A.value) else None
        imB = load_tiff(Path(dd_B.value)) if dd_B.value and os.path.exists(dd_B.value) else None

        # Load checkpoint C from pred_dir / cp_text
        cp_path = pred_dir / cp_text.value
        imC = load_tiff(cp_path) if cp_text.value and cp_path.exists() else None

        # Early warnings
        warn = []
        if imA is None: warn.append("A missing")
        if imB is None: warn.append("B missing")
        if imC is None: warn.append("Checkpoint missing / not found")

        with out:
            if warn:
                print("Warnings:", ", ".join(warn))
            # Decide contrast
            vmA = percentiles(imA) if imA is not None else (0,1)
            vmB = percentiles(imB) if imB is not None else (0,1)
            vmC = percentiles(imC) if imC is not None else (0,1)
            if joint_contrast.value and all(v is not None for v in (imA, imB, imC)):
                lo = min(vmA[0], vmB[0], vmC[0])
                hi = max(vmA[1], vmB[1], vmC[1])
                vmA = vmB = vmC = (lo, hi)

            # Plot
            ncols = 3
            fig, axs = plt.subplots(1, ncols, figsize=(5.6*ncols, 5.6))
            axs = np.atleast_1d(axs)

            if imA is not None:
                show_image(axs[0], imA, *vmA, title=f"{title_prefix} — A: {Path(dd_A.value).name}")
            else:
                axs[0].set_title("A: not found"); axs[0].axis("off")

            if imB is not None:
                show_image(axs[1], imB, *vmB, title=f"{title_prefix} — B: {Path(dd_B.value).name}")
            else:
                axs[1].set_title("B: not found"); axs[1].axis("off")

            if imC is not None:
                show_image(axs[2], imC, *vmC, title=f"{title_prefix} — Checkpoint: {cp_text.value}")
            else:
                axs[2].set_title("Checkpoint image not found"); axs[2].axis("off")

            fig.tight_layout()
            plt.show()

        info.value = (
            f"<b>A:</b> {Path(dd_A.value).name if dd_A.value else '—'} &nbsp; "
            f"<b>B:</b> {Path(dd_B.value).name if dd_B.value else '—'} &nbsp; "
            f"<b>Checkpoint:</b> {cp_text.value or '—'}"
        )

    # Wire up
    for w in (dd_A, dd_B, cp_text, joint_contrast):
        w.observe(lambda *_: draw(), names="value")
    refresh_btn.on_click(draw)

    controls = VBox([
        HTML(f"<b>{title_prefix} — alternative methods vs. checkpoint</b>"),
        HBox([dd_A, dd_B], layout=Layout(gap="10px", align_items="center")),
        HBox([cp_text, dd_suggest], layout=Layout(gap="10px", align_items="center")),
        HBox([joint_contrast, refresh_btn], layout=Layout(gap="12px", align_items="center")),
    ], layout=Layout(gap="6px"))

    display(VBox([controls, info, out]))
    draw()

# Predictions after N epochs

In [ ]:
# ===== Browse: K-K predictions =====
create_predictions_browser(PRED_DIRS["K-K"], element_label="K-K")

In [ ]:
# ===== Browse: Ir-L predictions =====
create_predictions_browser(PRED_DIRS["Ir-L"], element_label="Ir-L")

In [ ]:
# ===== Browse: Zn-K predictions =====
create_predictions_browser(PRED_DIRS["Zn-K"], element_label="Zn-K")

# Compare with alternative methods

In [ ]:
# ===== Compare: K (alt methods vs. checkpoint from K-K/predictions) =====
create_comparison_view(
    element_short="K",
    comp_dir=COMP_DIRS["K"],
    pred_dir=PRED_DIRS["K-K"],
    title_prefix="K"
)

In [ ]:
# ===== Compare: Ir (alt methods vs. checkpoint from Ir-L/predictions) =====
create_comparison_view(
    element_short="Ir",
    comp_dir=COMP_DIRS["Ir"],
    pred_dir=PRED_DIRS["Ir-L"],
    title_prefix="Ir"
)

In [ ]:
# ===== Compare: Zn (alt methods vs. checkpoint from Zn-K/predictions) =====
create_comparison_view(
    element_short="Zn",
    comp_dir=COMP_DIRS["Zn"],
    pred_dir=PRED_DIRS["Zn-K"],
    title_prefix="Zn"
)